# Neural Search Engine — Google Colab GPU Setup

**Before running anything:**
1. `Runtime` → `Change runtime type` → `T4 GPU` → Save
2. Run cells top to bottom, once per session

---
### Why this notebook exists
- **Local machine** (Windows, no GPU): `uv sync` with `torch-backend = "auto"` installs the CPU-only torch wheel.
- **Colab** (Linux, T4/A100 GPU): Colab pre-installs PyTorch with CUDA already compiled in.
  We don't reinstall torch — we just add the handful of packages Colab is missing
  and point Python at our Drive folder.

> **We never use `uv` in Colab.** `uv` is for your local dev environment.
> Colab has its own package manager (`pip`) and its torch build is hand-tuned for
> the Colab GPU. Replacing it with a uv-managed copy would break CUDA support.

---
## Step 1 — Verify you have a GPU
If this shows `CUDA available: False`, go to Runtime → Change runtime type → GPU.

In [ ]:
import torch
print('PyTorch version :', torch.__version__)
print('CUDA available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU             :', torch.cuda.get_device_name(0))
    print('VRAM            :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('\nNo GPU found — go to Runtime → Change runtime type → T4 GPU')

---
## Step 2 — Mount Google Drive

A browser pop-up will ask you to sign in and grant access.  
Your Drive is then available under `/content/drive/MyDrive/`.

**Where to put your project:**  
Upload / sync the `Neural_Search_Engine` folder to the root of your Drive so the path is:
```
My Drive/
└── Neural_Search_Engine/
    ├── src/
    │   ├── model.py      ← your BiEncoder
    │   └── ...
    ├── data/
    ├── notebooks/
    └── pyproject.toml
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Change this if your folder is in a sub-folder of Drive, e.g.:
# PROJECT_ROOT = '/content/drive/MyDrive/university/Neural_Search_Engine'
PROJECT_ROOT = '/content/drive/MyDrive/Neural_Search_Engine'

import os
assert os.path.isdir(PROJECT_ROOT), (
    f'Folder not found: {PROJECT_ROOT}\n'
    f'Upload Neural_Search_Engine to your Google Drive first.'
)
print('Project found at:', PROJECT_ROOT)

---
## Step 3 — Install missing packages

Colab already ships with: `torch`, `numpy`, `pandas`, `scipy`, `scikit-learn`, `tqdm`.

We only install what Colab does NOT have.

In [ ]:
# -q = quiet output; --upgrade ensures we get versions compatible with Colab's torch
!pip install -q \
    transformers>=4.41.0 \
    tokenizers>=0.19.0 \
    safetensors>=0.4.3 \
    rank-bm25 \
    pymupdf \
    gensim

print('Installation complete.')

---
## Step 4 — Add the project to Python's path

`sys.path.insert` makes Python find `src/model.py` as `from src.model import BiEncoder`.
This is equivalent to what `uv run` does on your local machine.

In [ ]:
import sys

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Quick sanity check
import importlib.util
spec = importlib.util.find_spec('src.model')
print('src.model found at:', spec.origin if spec else 'NOT FOUND — check PROJECT_ROOT')

---
## Step 5 — Run the BiEncoder on GPU

This is the same smoke test from `src/model.py`, but now running on the Colab T4 GPU.

In [ ]:
import torch
from src.model import BiEncoder

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

# Load model and move all weights to GPU
model = BiEncoder().to(DEVICE)
model.eval()

# Fake batch — same as Tura will send, but on GPU
batch_size, max_length = 4, 256
ids  = torch.randint(0, 30522, (batch_size, max_length)).to(DEVICE)
mask = torch.ones(batch_size, max_length, dtype=torch.long).to(DEVICE)
mask[0, -50:] = 0   # simulate padding on sample 0

with torch.no_grad():
    embeddings = model(ids, mask)

# Assertions
assert embeddings.shape == (batch_size, 768)
assert torch.allclose(embeddings.norm(p=2, dim=1),
                       torch.ones(batch_size, device=DEVICE), atol=1e-5)

print(f'Output shape : {tuple(embeddings.shape)}  ✓')
print(f'L2 norms     : {embeddings.norm(p=2, dim=1).tolist()}  ✓')
print(f'Tensor lives on: {embeddings.device}')

---
## Step 6 — Measure the GPU speedup (optional but useful for your report)

Run this after confirming the model works. It shows how much faster the GPU is
versus CPU for a realistic training batch size.

In [ ]:
import time

def benchmark(device_str: str, n_runs: int = 20):
    dev = torch.device(device_str)
    m = BiEncoder().to(dev)
    m.eval()
    ids_b  = torch.randint(0, 30522, (32, 256)).to(dev)   # batch_size=32
    mask_b = torch.ones(32, 256, dtype=torch.long).to(dev)

    # Warmup
    with torch.no_grad():
        for _ in range(3):
            _ = m(ids_b, mask_b)

    if device_str == 'cuda':
        torch.cuda.synchronize()

    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(n_runs):
            _ = m(ids_b, mask_b)

    if device_str == 'cuda':
        torch.cuda.synchronize()

    elapsed = (time.perf_counter() - start) / n_runs
    return elapsed * 1000   # ms per forward pass

cpu_ms = benchmark('cpu')
print(f'CPU : {cpu_ms:.1f} ms / forward pass (batch=32, seq=256)')

if torch.cuda.is_available():
    gpu_ms = benchmark('cuda')
    print(f'GPU : {gpu_ms:.1f} ms / forward pass (batch=32, seq=256)')
    print(f'Speedup: {cpu_ms / gpu_ms:.1f}x')

---
## Cheat-Sheet: Local vs Colab

| | Local (Windows, no GPU) | Google Colab (T4 GPU) |
|---|---|---|
| **Run code** | `uv run python src/model.py` | This notebook |
| **Install deps** | `uv sync` (reads `pyproject.toml`) | `!pip install transformers ...` |
| **torch version** | `torch+cpu` (auto-selected) | Pre-installed `torch+cu...` |
| **Project location** | `C:/Users/Enele/Desktop/Neural_Search_Engine` | `/content/drive/MyDrive/Neural_Search_Engine` |
| **Import project** | Works automatically (uv sets path) | `sys.path.insert(0, PROJECT_ROOT)` |
| **Saving changes** | Normal file system | Edit files in Drive — changes are live immediately |

### Workflow for training runs
1. Write and debug code **locally** (fast iteration, no GPU needed for shape checks)
2. Push changes to Drive (Sync or copy the updated `.py` file)
3. Open **this notebook** in Colab, re-run Step 5 onwards
4. Run the training loop on GPU
5. Save model checkpoint back to Drive: `torch.save(model.state_dict(), PROJECT_ROOT + '/checkpoints/epoch1.pt')`